# Plot solute profiles

In [ ]:
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
from min3p.output import read_min3p_sequence
from byte_util.util import all_sites
from byte_util.met_forcing_plots import plot_profiles, plot_exchange_profiles, profiles_to_plot

s3_base_path = 's3://carbonplan-carbon-removal/ew-workflows-data/min3p'
s3_input_path = f'{s3_base_path}/input-data/processed-data'
soil_pco2 = pd.read_csv(f'{s3_input_path}/soil_pco2.csv', index_col=0)

In [ ]:
# Read in soil physical and chemical parameters
s3_base_path = 's3://carbonplan-carbon-removal/ew-workflows-data/min3p'
s3_input_path = f'{s3_base_path}/input-data/processed-data'

soil_phys = pd.read_parquet(f'{s3_input_path}/soil_physical_parameters.parquet')
soil_chem = pd.read_parquet(f'{s3_input_path}/soil_chemical_parameters.parquet')

In [ ]:
fig, ax = plt.subplots(2, len(all_sites), figsize=(12, 8), sharey=True, sharex='col', tight_layout=True)

for i, site in enumerate(all_sites):
    sim_folder = Path(f'../min3p_runs/{site}/spinup')
    gsg_file = sim_folder / 'spinup_1.gsg'
    if not gsg_file.exists():
        continue
    gsg, gsg_cols, timesteps = read_min3p_sequence(gsg_file)
    if timesteps[-1] > 36000:
        mask = timesteps > 3649
        timesteps = timesteps[mask]
        gsg = gsg[mask]
    colors = plt.colormaps['cividis'](np.linspace(0, 1, gsg.shape[0]))

    z = gsg[0, 2, 0]
    depth = z.max() - z  # Convert z (elevation) to depth

    for j, timestep in enumerate(timesteps):
        label = f'{timestep/365:.0f} y' if (j+1) % 2 == 0 else None

        ax[0, i].plot(gsg[j, gsg_cols.index('co2(g)'), 0], depth, color=colors[j], label=label, ls='--')
        ax[1, i].plot(gsg[j, gsg_cols.index('co2(g)'), 1], depth, color=colors[j], label=label, ls='-')

    ax[0, i].set(title=f'{site}:\nMacropores', ylim=(depth.max(), depth.min()))
    ax[1, i].set(xlabel='pCO2 (atm)', title='Matrix', ylim=(depth.max(), depth.min()))

    ax[0, i].axvline(10**soil_pco2.loc[site, 'soilco2_log10atm'], color='k', linestyle='--')
    ax[1, i].axvline(10**soil_pco2.loc[site, 'soilco2_log10atm'], color='k', linestyle='--')

    ax[0, 0].set(ylabel='Depth')
    ax[1, 0].set(ylabel='Depth')
ax[0, -1].legend()
ax[-1, -1].legend()

In [ ]:
fig, ax = plt.subplots(2, len(all_sites), figsize=(12, 10), sharey=True, sharex='col', tight_layout=True)

for i, site in enumerate(all_sites):
    sim_folder = f'../min3p_runs/{site}/spinup'
    sim_name = 'spinup'
    if not Path(f'{sim_folder}/{sim_name}_1.gsc').exists():
        continue
    gsc, gsc_cols, timesteps = read_min3p_sequence(f'{sim_folder}/{sim_name}_1.gsc')
    if timesteps[-1] > 36000:
        mask = timesteps > 3649
        timesteps = timesteps[mask]
        gsc = gsc[mask]

    colors = plt.colormaps['cividis'](np.linspace(0, 1, gsc.shape[0]))
    z = gsc[0, 2, 0]
    depth = z.max() - z  # Convert z (elevation) to depth

    for j, timestep in enumerate(timesteps):
        label = f'{timestep/365:.0f} y' if (j+1) % 2 == 0 else None
        for d, (domain, ls) in enumerate([('Macropores', '--'), ('Matrix', '-')]):
            ax[d, i].plot(-np.log10(gsc[j, gsc_cols.index('h+1'), d]), depth, color=colors[j], label=label, ls=ls)
            ax[d, i].set(title=site, ylim=(depth.max(), depth.min()))
            title = domain if d == 1 else f'{site}:\n{domain}'
            ax[d, i].set(title=title, ylim=(depth.max(), depth.min()))
            if d == 1:
                ax[d, i].set(xlabel='pH')

    # Add expected pH
    hzns = ['A', 'B', 'C']
    for hzn in hzns:
        expected_ph = soil_chem.loc[(site, hzn), 'pH']
        ax[1, i].plot([expected_ph, expected_ph],
                   soil_phys.loc[(site, hzn), ['top_m', 'bottom_m']],
                   color='k', linestyle='--')

for d in range(2):
    ax[d, 0].set(ylabel='Depth')
    ax[d, -1].legend()

In [ ]:
replot = True

sim_folder = Path('../min3p_runs')
pdf_file = Path('plots') / 'SpinupProfiles.pdf'

if replot:
    with PdfPages(pdf_file) as pdf:
        sim_name = 'spinup'

        ## Add pCO2 plot
        fig, ax = plt.subplots(2, len(all_sites), figsize=(12, 8), sharey=True, sharex='col', tight_layout=True)
        for i, site in enumerate(all_sites):
            site_folder = sim_folder / site / 'spinup'
            if not (site_folder / f'{sim_name}_1.gsg').exists():
                continue
            gsg, gsg_cols, timesteps = read_min3p_sequence(f'{sim_name}_1.gsg', folder=site_folder)
            if timesteps[-1] > 36000:
                mask = timesteps > 3649
                timesteps = timesteps[mask]
                gsg = gsg[mask]
            colors = plt.colormaps['cividis'](np.linspace(0, 1, gsg.shape[0]))
            z = gsg[0, 2, 0]
            depth = z.max() - z  # Convert z (elevation) to depth
            for j, timestep in enumerate(timesteps):
                label = f'{timestep/365:.0f} y' if (j+1) % 2 == 0 else None
                ax[0, i].plot(gsg[j, gsg_cols.index('co2(g)'), 0], depth, color=colors[j], label=label, ls='--')
                ax[1, i].plot(gsg[j, gsg_cols.index('co2(g)'), 1], depth, color=colors[j], label=label, ls='-')
            ax[0, i].set(title=f'{site}:\nMacropores', ylim=(depth.max(), depth.min()))
            ax[1, i].set(xlabel='pCO2 (atm)', title='Matrix', ylim=(depth.max(), depth.min()))
            ax[0, i].axvline(10**soil_pco2.loc[site, 'soilco2_log10atm'], color='k', linestyle='--')
            ax[1, i].axvline(10**soil_pco2.loc[site, 'soilco2_log10atm'], color='k', linestyle='--')
        for d in range(2):
            ax[d, 0].set(ylabel='Depth')
            ax[d, -1].legend()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

        ## Add pH plot
        fig, ax = plt.subplots(2, len(all_sites), figsize=(12, 8), sharey=True, sharex='col', tight_layout=True)
        for i, site in enumerate(all_sites):
            site_folder = sim_folder / site / 'spinup'
            if not (site_folder / f'{sim_name}_1.gsc').exists():
                continue
            gsc, gsc_cols, timesteps = read_min3p_sequence(f'{sim_name}_1.gsc', folder=site_folder)
            if timesteps[-1] > 36000:
                mask = timesteps > 3649
                timesteps = timesteps[mask]
                gsc = gsc[mask]
            colors = plt.colormaps['cividis'](np.linspace(0, 1, gsc.shape[0]))
            for j, timestep in enumerate(timesteps):
                label = f'{timestep/365:.0f} y' if (j+1) % 2 == 0 else None
                for d, (domain, ls) in enumerate([('Macropores', '--'), ('Matrix', '-')]):
                    ax[d, i].plot(-np.log10(gsc[j, gsc_cols.index('h+1'), d]), depth, color=colors[j], label=label, ls=ls)
                    ax[d, i].set(title=site, ylim=(depth.max(), depth.min()))
                    title = domain if d == 1 else f'{site}:\n{domain}'
                    ax[d, i].set(title=title, ylim=(depth.max(), depth.min()))
                    if d == 1:
                        ax[d, i].set(xlabel='pH')
            # Add expected pH
            hzns = ['A', 'B', 'C']
            for hzn in hzns:
                expected_ph = soil_chem.loc[(site, hzn), 'pH']
                ax[1, i].plot([expected_ph, expected_ph], soil_phys.loc[(site, hzn), ['top_m', 'bottom_m']],
                              color='k', linestyle='--')
        for d in range(2):
            ax[d, 0].set(ylabel='Depth')
            ax[d, -1].legend()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

        ## Add alkalinity plot
        fig, ax = plt.subplots(2, len(all_sites), figsize=(12, 8), sharey=True, sharex='col', tight_layout=True)
        for i, site in enumerate(all_sites):
            site_folder = sim_folder / site / 'spinup'
            if not (site_folder / f'{sim_name}_1.gsc').exists():
                continue
            gsm, gsm_cols, timesteps = read_min3p_sequence(f'{sim_name}_1.gsm', folder=site_folder)
            if timesteps[-1] > 36000:
                mask = timesteps > 3649
                timesteps = timesteps[mask]
                gsm = gsm[mask]
            colors = plt.colormaps['cividis'](np.linspace(0, 1, gsm.shape[0]))
            for j, timestep in enumerate(timesteps):
                label = f'{timestep/365:.0f} y' if (j+1) % 2 == 0 else None
                for d, (domain, ls) in enumerate([('Macropores', '--'), ('Matrix', '-')]):
                    ax[d, i].plot(1000*gsm[j, gsm_cols.index('Alk [eq/L]'), d], depth, color=colors[j], label=label, ls=ls)
                    title = domain if d == 1 else f'{site}:\n{domain}'
                    ax[d, i].set(title=title, ylim=(depth.max(), depth.min()))
                    if d == 1:
                        ax[d, i].set(xlabel='Alkalinity (meq/L)')
        for d in range(2):
            ax[d, 0].set(ylabel='Depth')
            ax[d, -1].legend()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

        ## Add all other plots
        for site in tqdm(all_sites):
            site_folder = sim_folder / site / 'spinup'
            if not (site_folder / f'{sim_name}_1.gsc').exists():
                continue

            # Add site name
            fig, ax = plt.subplots(figsize=(8, 5))
            ax.annotate(site, (0.5, 0.5), fontsize=32, ha='center', va='center')
            ax.xaxis.set_visible(False)
            ax.yaxis.set_visible(False)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

            for page_no, plot_info in tqdm(profiles_to_plot.items(), leave=False, desc=f'Generating plots for {site}'):
                if plot_info['vars'] == ['plot_exchange_profiles']:
                    fig, ax = plot_exchange_profiles(site, 'spinup', itime=-1, dualperm=True)
                else:
                    fig, ax = plot_profiles(plot_info['vars'], site, 'spinup', starting_timestep=3649, dualperm=True)

                # Add expected pH and CO2
                if 'h+1' in plot_info['vars']:
                    axidx = plot_info['vars'].index('h+1')
                    for hzn in ['A', 'B', 'C']:
                        expected_ph = soil_chem.loc[(site, hzn), 'pH']
                        ax[1, axidx].plot([expected_ph, expected_ph], soil_phys.loc[(site, hzn), ['top_m', 'bottom_m']],
                                          color='k', linestyle='--')
                if 'co2(g)' in plot_info['vars']:
                    axidx = plot_info['vars'].index('co2(g)')
                    for d in range(2):
                        ax[d, axidx].axvline(10**soil_pco2.loc[site, 'soilco2_log10atm'], color='k', linestyle='--')

                fig.suptitle(f'{site}: {plot_info['title']}')
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)

# Check a few major profiles
### Water content and root water uptake

In [ ]:
site = 'HoustonBlack'
vars_to_plot = ['h_w', 'theta_a', 'q_root']
fig, ax = plot_profiles(vars_to_plot, site, 'spinup', dualperm=True)
fig.suptitle('Water content and root uptake')

### Tracer profiles

In [ ]:
vars_to_plot = ['psi01', 'psi02', 'cl-1']
fig, ax = plot_profiles(vars_to_plot, site, 'spinup', dualperm=True)
fig.suptitle('Tracer profiles')

### pH and CO2 respiration
Ideally within a "reasonable" range

In [ ]:
vars_to_plot = ['h+1', 'hco3-', 'co2(g)', 'co2_resp', 'Alk [eq/L]']
fig, ax = plot_profiles(vars_to_plot, site, 'spinup', dualperm=True)
# Add field pH
for d in range(2):
    offset = d*len(vars_to_plot)
    hzns = ['A', 'B', 'C']
    for hzn in hzns:
        expected_ph = soil_chem.loc[(site, hzn), 'pH']
        ax[d, 0].plot([expected_ph, expected_ph],
                      soil_phys.loc[(site, hzn), ['top_m', 'bottom_m']],
                      color='k', linestyle='--')
    ax[d, 2].axvline(10**soil_pco2.loc[site, 'soilco2_log10atm'], color='k', linestyle='--')
    ax[d, 3].set(xlim=[0, 0.0014], xticks=[0, 0.001])
    ax[d, 4].set(xlabel='Alk (meq/L)')
fig.suptitle('pH, alkalinity, and CO2 respiration')

In [ ]:
vars_to_plot = ['h+1', 'hco3-', 'co2(g)', 'co2_resp', 'Alk [eq/L]']
fig, ax = plot_profiles(vars_to_plot, site, 'spinup', dualperm=True)

### Compare gas tracers

In [ ]:
vars_to_plot = ['gtr(aq)', 'gtr(g)', 'co2(g)', 'p_t_o_t', 'co2_resp']
fig, ax = plot_profiles(vars_to_plot, site, 'spinup', dualperm=True)
for d in range(2):
    ax[d, 2].axvline(10**soil_pco2.loc[site, 'soilco2_log10atm'], color='k', linestyle='--')
    ax[d, -1].set(xlim=[0, 0.0014], xticks=[0, 0.001])
fig.suptitle('CO2 and gas tracer')

### Carbonate speciation

Ideally, `hco3-` should dominate over `co3-2`

In [ ]:
vars_to_plot = ['h2co3aq', 'hco3-', 'co3-2']
fig, ax = plot_profiles(vars_to_plot, site, 'spinup', dualperm=True)
fig.suptitle('Carbonate speciation')

### Major cation profiles

Ideally, background Ca, Mg, Na, and K should remain stable during spin-up before adding feedstock.

In [ ]:
vars_to_plot = ['mg+2', 'ca+2', 'na+1', 'k+1', 'al+3']
fig, ax = plot_profiles(vars_to_plot, 'Cecil', 'spinup')
fig.suptitle('Major cation profiles')

### Silica system

In [ ]:
vars_to_plot = ['h4sio4', 'h3sio4-', 'h2sio4-2', 'sio2_vol']
fig, ax = plot_profiles(vars_to_plot, site, 'spinup')
fig.suptitle('Silica system')

### K-feldspar dissolution and gibbsite precipitation

In [ ]:
vars_to_plot = ['al+3', 'k_feld_vol', 'gibbsite_vol']
fig, ax = plot_profiles(vars_to_plot, site, 'spinup')
ax[1].set(xlim=(0, ax[1].get_xlim()[1]*1.1))
fig.suptitle('K-feldspar gibbsite system')

In [ ]:
fig, ax = plot_exchange_profiles(site, 'spinup', itime=-1)
fig.suptitle('Base saturation and exchangeable-cation composition')

In [ ]:
vars_to_plot = ['base_sat', 'ca_exch', 'mg_exch', 'al_exch', 'h_exch']
fig, ax = plot_profiles(vars_to_plot, site, 'spinup')
fig.suptitle('Cation exchange')

### Calcite buffering profiles
Is calcite dissolving or precipitating during spinup? Is it controlling Ca and pH?

In [ ]:
vars_to_plot = ['h+1', 'ca+2', 'hco3-', 'co3-2']
fig, ax = plot_profiles(vars_to_plot, site, 'spinup')
fig.suptitle('Calcite buffering')

vars_to_plot = ['calcite_neutral', 'calcite_base', 'calcite_acid',
                'calcite_SI', 'calcite_vol']
fig, ax = plot_profiles(vars_to_plot, site, 'spinup')
fig.suptitle('Calcite dissolution')

In [ ]:
fig, ax = plot_profiles(['k_feld_neutral', 'k_feld_acid', 'k_feld_base', 'k_feld_SI', 'k_feld_vol'], site, 'spinup')
fig.suptitle('K-feldspar dissolution')

In [ ]:
fig, ax = plot_profiles(['namont_neutral', 'namont_acid', 'namont_base', 'namont_SI', 'namont_vol'], site, 'spinup')
fig.suptitle('Na-montmorillonite dissolution')

In [ ]:
fig, ax = plot_profiles(['forst_neutral', 'forst_acid', 'forst_SI', 'forst_vol'], site, 'spinup')
fig.suptitle(f'{site}: Forsterite dissolution')